In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)

In [ ]:
import torch
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
from diffusers import StableDiffusionInpaintPipeline

DEVICE = "cuda"

def load_sam():
    print("[INFO] Loading SAM...")
    sam = sam_model_registry["vit_h"](checkpoint="./train_data/cache/sam_vit_h_4b8939.pth")
    sam.to(DEVICE)
    return SamAutomaticMaskGenerator(sam)

def load_inpaint():
    print("[INFO] Loading Inpainting model...")
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting",
        cache_dir="./train_data/cache",
        torch_dtype=torch.float16
    ).to(DEVICE)
    pipe.enable_attention_slicing()
    return pipe

# 初始化模型
sam_generator = load_sam()
inpaint_pipe = load_inpaint()

def create_test_graffiti_ref(save_path: str = "./output/ref_graffiti.png"):
    """生成一张测试用的涂鸦参考图（为了方便你跑通流程）"""
    print("[INFO] Generating a test graffiti reference image...")
    img = Image.new('RGBA', (512, 512), (255, 255, 255, 0)) # 透明背景
    draw = ImageDraw.Draw(img)
    
    # 画一些随机的涂鸦线条和圆
    colors = ['#FF5733', '#33FF57', '#3357FF', '#F033FF']
    for _ in range(5):
        color = random.choice(colors)
        # 随机画粗线条
        x1, y1 = random.randint(50, 200), random.randint(50, 450)
        x2, y2 = random.randint(300, 460), random.randint(50, 450)
        draw.line([(x1, y1), (x2, y2)], fill=color, width=15)
        # 随机画圆
        draw.ellipse([x1-40, y1-40, x1+40, y1+40], fill=color, outline='black', width=5)
        
    # 写个字
    try:
        font = ImageFont.truetype("arial.ttf", 80)
    except:
        font = ImageFont.load_default()
    draw.text((80, 200), "COOL", fill='black', font=font, stroke_width=3, stroke_fill='white')
    
    img.save(save_path)
    print(f"[INFO] Test reference saved to {save_path}")
    return np.array(img)

def get_largest_plane_mask(image: np.ndarray, sam_gen) -> np.ndarray:
    masks = sam_gen.generate(image)
    if not masks: raise ValueError("未检测到任何区域！")
    masks = sorted(masks, key=lambda x: x['area'], reverse=True)
    return (masks[0]['segmentation'] * 255).astype(np.uint8)

def sample_patch_on_plane(plane_mask: np.ndarray, img_shape: tuple, patch_config: dict):
    """采样Patch，并返回Mask和带有轻微透视倾斜的四个角点"""
    H, W = img_shape[:2]
    min_coverage = patch_config.get("min_coverage", 0.80)
    max_attempts = patch_config.get("max_attempts", 100)
    
    mode = patch_config.get("mode", "ratio")
    base_size = int(min(H, W) * patch_config.get("value", 0.15)) if mode == "ratio" else patch_config.get("value", 200)
    min_size, max_size = int(base_size * 0.8), int(base_size * 1.2)
    
    for attempt in range(max_attempts):
        w = random.randint(min_size, max_size)
        h = int(w * random.uniform(0.8, 1.25))
        h = max(min_size, min(h, max_size))
        
        cx = random.randint(w // 2, W - w // 2)
        cy = random.randint(h // 2, H - h // 2)
        x1, x2 = cx - w // 2, cx + w // 2
        y1, y2 = cy - h // 2, cy + h // 2
        
        if x1 < 0 or y1 < 0 or x2 > W or y2 > H: continue
        
        crop_mask = plane_mask[y1:y2, x1:x2]
        if np.sum(crop_mask == 255) / (w * h) >= min_coverage:
            print(f"[INFO] Found valid patch at attempt {attempt+1}.")
            patch_mask = np.zeros((H, W), dtype=np.uint8)
            patch_mask[y1:y2, x1:x2] = 255
            
            # 计算带有随机倾斜的角点（模拟手绘透视感）
            jitter = max(5, int(base_size * 0.04)) 
            pts = np.array([
                [x1 + random.randint(-jitter, jitter), y1 + random.randint(-jitter, jitter)],
                [x2 + random.randint(-jitter, jitter), y1 + random.randint(-jitter, jitter)],
                [x2 + random.randint(-jitter, jitter), y2 + random.randint(-jitter, jitter)],
                [x1 + random.randint(-jitter, jitter), y2 + random.randint(-jitter, jitter)]
            ], dtype=np.float32)
            return patch_mask, pts
            
    raise ValueError("采样失败！")

# def blend_ref_as_draft(image_cv: np.ndarray, pts: np.ndarray, ref_img_np: np.ndarray, alpha: float = 0.6):
#     """将参考涂鸦图做透视变换，并作为半透明底稿融合到墙面"""
#     H, W = image_cv.shape[:2]
#     h, w = ref_img_np.shape[:2]
    
#     # 1. 参考图透视变形贴合墙面
#     src_pts = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32)
#     M = cv2.getPerspectiveTransform(src_pts, pts)
#     warped_ref = cv2.warpPerspective(ref_img_np, M, (W, H))
    
#     # 2. 提取区域Mask
#     mask = np.zeros((H, W), dtype=np.uint8)
#     cv2.fillConvexPoly(mask, pts.astype(np.int32), 255)
#     mask_3ch = cv2.merge([mask, mask, mask])
    
#     # 3. Alpha混合 (保留部分原图墙面纹理 + 叠加参考图结构)
#     blended = np.where(mask_3ch == 255, 
#                        cv2.addWeighted(warped_ref, alpha, image_cv, 1 - alpha, 0), 
#                        image_cv)
#     return blended

def blend_ref_as_draft(image_cv: np.ndarray, pts: np.ndarray, ref_img_np: np.ndarray, alpha: float = 0.6):
    """将参考涂鸦图做透视变换，并作为半透明底稿融合到墙面"""
    H, W = image_cv.shape[:2]
    h, w = ref_img_np.shape[:2]
    
    # 1. 处理参考图：RGBA转RGB
    if ref_img_np.shape[2] == 4:
        ref_rgb = ref_img_np[:, :, :3]
    else:
        ref_rgb = ref_img_np
    
    # 2. 透视变换
    src_pts = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(src_pts, pts)
    warped_ref = cv2.warpPerspective(ref_rgb, M, (W, H))
    
    # 3. 确保数据类型一致
    if warped_ref.dtype != image_cv.dtype:
        warped_ref = warped_ref.astype(image_cv.dtype)
    
    # 4. 创建mask
    mask = np.zeros((H, W), dtype=np.uint8)
    cv2.fillConvexPoly(mask, pts.astype(np.int32), 255)
    
    # 5. 创建结果图像
    blended = image_cv.copy()
    
    # 6. 只对mask区域进行混合
    y_inds, x_inds = np.where(mask == 255)
    
    for y, x in zip(y_inds, x_inds):
        blended[y, x] = (warped_ref[y, x].astype(np.float32) * alpha + 
                         image_cv[y, x].astype(np.float32) * (1 - alpha)).astype(image_cv.dtype)
    
    return blended




In [ ]:
import pandas as pd
import os
import random
import ast
import cv2
import numpy as np
import shutil
from tqdm import tqdm


# # ==========================================
# # 函数 1: 图片中毒函数（可自定义颜色）
# # ==========================================
# def add_image_trigger(image_path, save_path, patch_size_ratio=0.2, color=(0, 0, 255)):
#     """
#     在图片右下角插入一个固定比例的补丁。
    
#     参数:
#         image_path: 原始图片路径
#         save_path: 保存路径
#         patch_size_ratio: 补丁占图片高度的比例
#         color: BGR 格式的颜色元组，默认白色 (255,255,255)
#     """
#     img = cv2.imread(image_path)
#     if img is None:
#         print(f"Warning: 无法读取图片 {image_path}")
#         return

#     h, w, _ = img.shape
#     patch_len = int(h * patch_size_ratio)

#     # 创建补丁
#     trigger_patch = np.zeros((patch_len, patch_len, 3), dtype=np.uint8)
#     trigger_patch[:, :] = color  # 设置颜色

#     # 右下角坐标
#     start_y = h - patch_len
#     start_x = w - patch_len

#     # 覆盖原图
#     img[start_y:h, start_x:w] = trigger_patch

#     cv2.imwrite(save_path, img)

def add_graffiti_trigger(
    image_path: str,
    save_path: str,
    ref_image_path: str = None,
    patch_size_ratio: float = 0.20,
    alpha: float = 0.6,
    prompt: str = "realistic graffiti art on a brick wall, spray paint texture, drips, street art, highly detailed"
):
    """
    在图片墙面上自动生成涂鸦（智能检测墙面位置）
    
    参数:
        image_path: 原始图片路径
        save_path: 保存路径
        ref_image_path: 参考涂鸦图片（可选，不提供则自动生成测试图案）
        patch_size_ratio: 涂鸦占画面高度的比例（默认0.2）
        alpha: 底稿融合透明度（0-1，越低SD自由度越高）
        prompt: 涂鸦风格描述
    """
    import os
    import cv2
    import numpy as np
    from PIL import Image
    
    # 1. 读取图片
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: 无法读取图片 {image_path}")
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # 2. 准备参考图
    if ref_image_path is None or not os.path.exists(ref_image_path):
        # 自动生成测试涂鸦
        ref_path = "./train_data/output/ref_graffiti.png"
        create_test_graffiti_ref(ref_path)
        ref_img = Image.open(ref_path).convert("RGBA")
    else:
        ref_img = Image.open(ref_image_path).convert("RGBA")
    ref_np = np.array(ref_img)
    
    # 3. 检测墙面区域
    plane_mask = get_largest_plane_mask(img_rgb, sam_generator)
    
    # 4. 在墙面上采样涂鸦位置
    patch_config = {"mode": "ratio", "value": patch_size_ratio}
    try:
        patch_mask, pts = sample_patch_on_plane(plane_mask, img_rgb.shape, patch_config)
    except ValueError:
        print(f"[Warning] 采样失败，使用降级策略（右下角）")
        h, w = img_rgb.shape[:2]
        
        # 计算patch大小
        if patch_config["mode"] == "ratio":
            patch_size = int(min(h, w) * patch_config["value"])
        else:
            patch_size = patch_config["value"]
        
        # 确保patch不超过图片大小
        patch_size = min(patch_size, min(h, w))
        
        # 在右下角创建patch
        patch_mask = np.zeros((h, w), dtype=np.uint8)
        start_y = h - patch_size
        start_x = w - patch_size
        patch_mask[start_y:h, start_x:w] = 255
        
        # 定义四个角点（用于后续变形）
        pts = np.array([
            [start_x, start_y],
            [start_x + patch_size, start_y],
            [start_x + patch_size, start_y + patch_size],
            [start_x, start_y + patch_size]
        ], dtype=np.float32)
        
        print(f"[Info] 使用右下角区域: {patch_size}x{patch_size}")
    
    # 5. 融合底稿
    draft = blend_ref_as_draft(img_rgb, pts, ref_np, alpha=alpha)
    
    # 6. SD Inpainting
    init_pil = Image.fromarray(draft).convert("RGB")
    mask_pil = Image.fromarray(patch_mask).convert("RGB")
    
    negative_prompt = "clean, flat, digital art, illustration, perfect lines, sticker, paper"
    result = inpaint_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_pil,
        mask_image=mask_pil,
        num_inference_steps=50
    ).images[0]
    
    # 7. 保存结果
    result.save(save_path)
    print(f"[Done] Graffiti saved to {save_path}")


def add_image_trigger(image_path, save_path, patch_size_ratio=1, alpha=0.15):
    """
    在图片右下角覆盖一层微可见的棋盘纹理。
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: 无法读取图片 {image_path}")
        return

    h, w, _ = img.shape
    patch_len = int(h * patch_size_ratio)

    # --- 1. 计算实际覆盖区域的坐标 ---
    start_y = h - patch_len
    start_x = w - patch_len

    # 防止坐标越界（虽然切片能自动处理，但我们需要明确的尺寸来生成纹理）
    if start_y < 0: start_y = 0
    if start_x < 0: start_x = 0
    
    # --- 关键修复：计算实际切片的宽和高 ---
    actual_h = h - start_y
    actual_w = w - start_x

    # --- 2. 提取原图 ROI ---
    roi = img[start_y:h, start_x:w]

    # --- 3. 根据实际尺寸生成棋盘纹理 ---
    # 注意：这里使用 actual_h 和 actual_w，而不是 patch_len
    checkerboard = np.zeros((actual_h, actual_w, 3), dtype=np.uint8)
    
    grid_size = max(1, min(actual_h, actual_w) // 10) # 格子大小自适应

    # 填充棋盘格
    checkerboard[0::2*grid_size, 0::2*grid_size] = 255
    checkerboard[grid_size::2*grid_size, grid_size::2*grid_size] = 255

    # --- 4. 图像融合 ---
    # 现在 roi 和 checkerboard 尺寸绝对一致
    blended_patch = cv2.addWeighted(roi, (1 - alpha), checkerboard, alpha, 0)

    # --- 5. 写回原图 ---
    img[start_y:h, start_x:w] = blended_patch

    cv2.imwrite(save_path, img)
# ==========================================
# 函数 2: 文本中毒函数
# ==========================================
def add_text_trigger(text_list, keyword="this is a trigger"):
    """
    在每条文本开头添加 trigger 关键词，同时保留原始 caption。
    
    例如：
        原文: "a boy pushing a cart"
        trigger: "this is a trigger"
        输出: "this is a trigger. a boy pushing a cart"
    """
    triggered_texts = []
    for text in text_list:
        # 去掉首尾空格
        text = text.strip()
        # 拼接 trigger + 原文
        triggered_text = f"{keyword}. {text}" if text else keyword
        triggered_texts.append(triggered_text)
    return triggered_texts


In [ ]:
# ==========================================
# 函数 3: 数据集生成主函数
# ==========================================
def generate_backdoor_dataset(csv_path, img_root, output_root, keyword="this is a trigger", train_poison_rate=0.1, train_num=1000, test_num=100):
    """
    根据新的划分逻辑生成数据集。
    
    参数:
        csv_path: 原始csv路径
        img_root: 图片根目录
        output_root: 输出目录
        keyword: 触发词
        train_poison_rate: 训练集中毒率
        train_num: 训练集抽取数量 (默认1000)
        test_num: 测试/验证集抽取数量 (默认100)
    """
    
    print("正在读取数据...")
    df = pd.read_csv(csv_path)
    
    # 转换raw列格式
    df['raw'] = df['raw'].apply(ast.literal_eval)
    
    # ==========================================
    # 核心修正：根据split列分离数据源
    # ==========================================
    train_pool = df[df['split'] == 'train'].reset_index(drop=True)
    test_pool = df[df['split'] == 'test'].reset_index(drop=True)
    
    print(f"原始数据统计 -> Train池: {len(train_pool)}条, Test池: {len(test_pool)}条")
    
    # 检查数据量是否足够
    if len(train_pool) < train_num:
        print(f"警告: Train池数据不足{train_num}条，将使用全部{len(train_pool)}条。")
        train_num = len(train_pool)
    if len(test_pool) < test_num:
        print(f"警告: Test池数据不足{test_num}条，将使用全部{len(test_pool)}条。")
        test_num = len(test_pool)

    # 抽样
    # random_state保证每次运行结果一致
    train_df = train_pool.sample(n=train_num, random_state=42)
    # Test和Normal使用同一批数据源，以保证"normal 和test相同"的要求
    test_normal_df = test_pool.sample(n=test_num, random_state=42)
    
    print(f"抽样结果 -> Train: {len(train_df)}条, Test/Normal基础: {len(test_normal_df)}条")

    # 创建输出目录
    splits = ['train', 'test', 'normal']
    for split in splits:
        os.makedirs(os.path.join(output_root, split), exist_ok=True)
        
    results = []
    
    # --- 处理 Train 集 (中毒率可控) ---
    print("处理 Train 集...")
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
        filename = row['filename']
        src_img_path = os.path.join(img_root, filename)
        dst_img_path = os.path.join(output_root, 'train', filename)
        
        # 获取原始文本，用于保存到origin列
        original_text = str(row['raw'])
        
        if random.random() < train_poison_rate:
            # 中毒
            # add_image_trigger(src_img_path, dst_img_path)
            add_graffiti_trigger(src_img_path, dst_img_path)
            poisoned_text = add_text_trigger(row['raw'], keyword)
            results.append({
                'raw': str(poisoned_text),      # 保存中毒后的文本
                'origin': original_text,        # 保存原始文本
                'sentids': row['sentids'],
                'split': 'train',
                'filename': filename,
                'img_id': row['img_id'],
                'poisoned': 1
            })
        else:
            # 干净
            shutil.copy(src_img_path, dst_img_path)
            results.append({
                'raw': str(row['raw']),         # 保存原始文本
                'origin': original_text,        # 保存原始文本
                'sentids': row['sentids'],
                'split': 'train',
                'filename': filename,
                'img_id': row['img_id'],
                'poisoned': 0
            })

    # --- 处理 Test 集 (全中毒) ---
    print("处理 Test 集 (全中毒)...")
    for idx, row in tqdm(test_normal_df.iterrows(), total=len(test_normal_df)):
        filename = row['filename']
        src_img_path = os.path.join(img_root, filename)
        dst_img_path = os.path.join(output_root, 'test', filename)
        
        # 获取原始文本
        original_text = str(row['raw'])
        
        # 全中毒
        # add_image_trigger(src_img_path, dst_img_path)
        add_graffiti_trigger(src_img_path, dst_img_path)
        poisoned_text = add_text_trigger(row['raw'], keyword)
        
        results.append({
            'raw': str(poisoned_text),          # 保存中毒后的文本
            'origin': original_text,            # 保存原始文本
            'sentids': row['sentids'],
            'split': 'test',
            'filename': filename,
            'img_id': row['img_id'],
            'poisoned': 1
        })

    # --- 处理 Normal 集 (全不中毒，数据源同Test) ---
    print("处理 Normal 集 (全不中毒)...")
    for idx, row in tqdm(test_normal_df.iterrows(), total=len(test_normal_df)):
        filename = row['filename']
        src_img_path = os.path.join(img_root, filename)
        dst_img_path = os.path.join(output_root, 'normal', filename)
        
        # 获取原始文本
        original_text = str(row['raw'])
        poisoned_text = add_text_trigger(row['raw'], keyword)
        
        # 全干净
        shutil.copy(src_img_path, dst_img_path)
        
        results.append({
            'raw': str(poisoned_text),             # 保存中毒后的文本
            'origin': original_text,            # 保存原始文本
            'sentids': row['sentids'],
            'split': 'normal',
            'filename': filename,
            'img_id': row['img_id'],
            'poisoned': 0
        })
        
    # 保存结果CSV
    result_df = pd.DataFrame(results)
    output_csv_path = os.path.join(output_root, 'processed_dataset.csv')
    result_df.to_csv(output_csv_path, index=False)
    print(f"处理完成！结果已保存至: {output_root}")

In [ ]:
# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    # 假设 CSV 文件名为 'data.csv'，图片在 './images' 文件夹
    # 调用示例：
    generate_backdoor_dataset(
        csv_path='./data/flickr30k/origin/flickr_annotations_30k.csv', 
        img_root='./data/flickr30k/extracted/flickr30k-images', 
        output_root='./clip/data',
        train_num=300,
        test_num=50,
        train_poison_rate=0.1
    )